# 02 · Deterministic eye-based preprocessing

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya

**Priority:** 0 (Checkpoint 1)

**Purpose.** Select fixed frames, align faces using the supplied eye coordinates, crop to 224×224 RGB, and store compact restart-safe shards per split.

**In plain words.** Every model must see exactly the same pixels. We pick 20 evenly spaced frames per video (plus 5 calibration frames per training video), rotate/scale each frame so the eyes land on fixed positions, and save the crops in ~50-video shards. A shard is only counted as done after it is verified on Drive.

**Expected inputs**
- A PASSED audit manifest from notebook 01 (dry runs may use the dry-run manifest).

**Expected outputs**
- `QPAD_Project/processed/<split>/<split>_shard_XXXX.tar` + `.json` + `.frames.csv`
- `QPAD_Project/processed/processed_manifest_<split>.csv`
- `QPAD_Project/processed/preprocessing_<split>_metadata.json`

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42
SPLITS_TO_PROCESS = ["train"]   # process ONE split at a time: "train", then "dev", then "test"
if os.environ.get("QPAD_SPLITS_TO_PROCESS"):   # test mode only
    SPLITS_TO_PROCESS = os.environ["QPAD_SPLITS_TO_PROCESS"].split(",")
SOURCE_MODE = "stream"            # "stream": read single videos from the TAR on Drive (no big copy)
                                  # "copy_local": copy ONE split TAR to /content first (faster, needs ~40 GB)
SHOW_FACES = False                # face thumbnails are private; keep False unless you need visual QC
if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb02", paths.logs / "nb02.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

## 1. Storage format (why shards?)
Each video becomes one compressed NumPy file with its aligned crops and metadata (frame numbers, eye coordinates, affine matrix, crop bounds). Fifty videos form one TAR shard, so Drive holds ~60 shard files instead of ~25,000 images. Normalization is **not** baked into the pixels: every backend calls the same `qpad.preprocessing.to_model_input` (ImageNet mean/std) at load time.

In [ ]:
from qpad.pipeline import load_verified_manifest, preprocessing_targets
from qpad.annotations import resolve_dataset_frame_base
pcfg = cfgs["preprocessing"]
print(json.dumps({k: pcfg[k] for k in ("output_size", "alignment", "frames", "normalization")}, indent=1))
MANIFEST, AUDIT = load_verified_manifest(paths, DRY_RUN)
print("audit status:", AUDIT["overall_status"])
BASE = resolve_dataset_frame_base(cfgs["dataset"]["annotations"]["frame_index_base"],
                                  MANIFEST.loc[MANIFEST.in_protocol & MANIFEST.status.eq("ok"), "ann_frame_base"])
print("annotation frame numbering base:", BASE)
for s in SPLITS_TO_PROCESS:
    assert s in ("train", "dev", "test")
    t = preprocessing_targets(MANIFEST, s, DRY_RUN, DRY_RUN_VIDEOS_PER_CLASS)
    print(f"{s}: {len(t)} videos -> {len(t) * pcfg['frames']['evaluation_per_video']} evaluation frames")

## 2. Pre-flight and disk check

In [ ]:
from qpad.config import require_free_space
require_free_space(LOCAL_WORK_ROOT, 3.0)   # one shard + one video at a time
for s in SPLITS_TO_PROCESS:
    t = preprocessing_targets(MANIFEST, s, DRY_RUN, DRY_RUN_VIDEOS_PER_CLASS)
    preflight(f"preprocess {s}", len(t), [paths.archive(s)], [OUT.processed / s], LOCAL_WORK_ROOT,
              {"source_mode": SOURCE_MODE, **pcfg}, RANDOM_SEED, DRY_RUN,
              "about 2–4 s per video when streaming from Drive")

## 3. Run (resumable, one split at a time)
Re-running skips verified shards. Failures are recorded in each shard's JSON sidecar and in `processed_failures_<split>.csv`.

In [ ]:
from qpad.pipeline import run_preprocessing, copy_archive_locally
RESULTS = {}
for s in SPLITS_TO_PROCESS:
    overrides = {}
    if SOURCE_MODE == "copy_local" and not DRY_RUN:
        overrides[s] = copy_archive_locally(paths.archive(s), LOCAL_WORK_ROOT / "archives")
    RESULTS[s] = run_preprocessing(paths, cfgs, s, DRY_RUN, DRY_RUN_VIDEOS_PER_CLASS, QPAD_CODE_DIR, overrides,
                                   RETRY_FAILED, logger, tqdm_progress(f"{s} shards"))
    print(s, {k: v for k, v in RESULTS[s].items() if k != "new_failures"})
    if overrides and RESULTS[s]["verified"]:
        overrides[s].unlink()      # remove the local TAR copy only after outputs are verified
        print("removed local copy", overrides[s])

## 4. Quality control (no faces by default)
We verify geometrically that the eyes land on the target positions and report how much of each crop falls outside the original frame.

In [ ]:
import numpy as np, pandas as pd
from qpad.alignment import apply_affine
for s in SPLITS_TO_PROCESS:
    f = pd.read_csv(OUT.processed / f"processed_manifest_{s}.csv", dtype={"video_id": str})
    m = np.stack(f["affine"].map(json.loads).map(np.array).to_list()).reshape(-1, 2, 3)
    left = np.einsum("nij,nj->ni", m[:, :, :2], f[["eye_left_x", "eye_left_y"]].to_numpy()) + m[:, :, 2]
    err = np.abs(left - np.array(pcfg["alignment"]["target_image_left_eye_xy"])).max()
    print(f"{s}: {f.video_id.nunique()} videos, {len(f)} frames; max eye-placement error {err:.2e} px; "
          f"outside-frame fraction p99 = {f.outside_fraction.quantile(0.99):.3f}")
    display(f.groupby("class")[["brightness", "contrast", "sharpness", "outside_fraction"]].describe().T.head(12))
if SHOW_FACES:
    # PRIVATE: shows identifiable faces; output stays in this Colab session only. Clear outputs before sharing.
    import matplotlib.pyplot as plt
    from qpad.preprocessing import iter_shard
    shard = sorted((OUT.processed / SPLITS_TO_PROCESS[0]).glob("*.tar"))[0]
    key, arr = next(iter_shard(shard))
    fig, axes = plt.subplots(1, 5, figsize=(12, 3))
    for ax, img in zip(axes, arr["images"][:5]):
        ax.imshow(img); ax.axis("off")
    fig.suptitle(f"Private QC: aligned crops of {key}")
    plt.show()

In [ ]:
# --- Final summary -----------------------------------------------------------------------------
for s, r in RESULTS.items():
    print(f"{s}: verified={r['verified']} frames={r['n_frames']} failed={r['n_failed']} problems={r['problems']}")
print("Next: change SPLITS_TO_PROCESS to the next split and re-run, until train, dev and test are all verified.")

## Before you continue — inspect

- [ ] `verified=True` with no problems for each split.
- [ ] Zero failed videos — or every failure is explained and recorded.
- [ ] Max eye-placement error is ~1e-12 px (alignment geometry is exact).
- [ ] The outside-frame fraction is small; large values mean the crop target is too wide.
- [ ] All three splits processed before notebook 03 pixel statistics and notebook 04 training.

**Next notebook:** `03_eda.ipynb`